# CASE 02 Statistical Analysis

가설은 EDA 핸드오프에서 미리 정한다. p값과 효과 크기를 함께 본다.

In [1]:
from pathlib import Path
import sys

CASE_NAME = "02_youth_migration_dynamics"
RAW_NAME = "2025_domestic_migration_statistics.xlsx"
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "cases" / CASE_NAME,
]
CASE_DIR = next(
    (path.resolve() for path in candidates if (path / "data" / "raw" / RAW_NAME).exists()),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(
        f"{RAW_NAME}를 찾지 못했습니다. 저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )
sys.path.insert(0, str(CASE_DIR / "src"))

from constants import RAW_FILE_NAME
from data_preparation import load_and_prepare
from data_quality import run_quality_checks
from kpi_segmentation import run_kpi_segmentation
from parse_official_tables import verify_source_file
from statistical_analysis import run_statistical_analysis

RAW = CASE_DIR / "data" / "raw" / RAW_FILE_NAME
source = verify_source_file(RAW)
prepared = load_and_prepare(RAW)
tables = prepared.tables
print(source["sha256"])
print("stale sheet present:", tables.workbook["has_stale_monthly_sheet"])

stats = run_statistical_analysis(tables, prepared)
stats.hypothesis_summary

FE066C40AAE0AE5C34C67947B404DC8943405C9BF0A7952DA09B0CF26D901E9D
stale sheet present: True


,hypothesis,p_value,significant_at_0.05,effect,statement
0,H1,9.536743e-07,True,1.000000,연도별로 짝지은 20-24세 이동률이 40-44세 이동률과 같다
1,H2,6.951311e-01,False,0.090909,2005-2025 청년(20-39) 이동자 비중이 연도와 상관없다
2,H3,1.080789e-02,True,0.600490,2025 시도 전체 순이동과 청년(20-39) 순이동이 상관없다
3,H4,1.033561e-04,True,0.771429,수도권 순이동(비수도권 대비)의 1990-2010 분포와 2011-2025 분포가 같다


In [2]:
stats.h1

{'n': 21,
 'statistic': 0.0,
 'p_value': 9.5367431640625e-07,
 'median_diff': 4.209961827999997,
 'rank_biserial': 1.0,
 'alpha': 0.05,
 'significant': True,
 'hypothesis': 'H1',
 'statement': '연도별로 짝지은 20-24세 이동률이 40-44세 이동률과 같다',
 'alternative': 'two-sided Wilcoxon signed-rank on yearly rate differences',
 'median_20_24': 19.254511523,
 'median_40_44': 14.447496085}

In [3]:
stats.h3

{'hypothesis': 'H3',
 'statement': '2025 시도 전체 순이동과 청년(20-39) 순이동이 상관없다',
 'n': 17,
 'spearman_rho': 0.6004901960784315,
 'p_value': 0.010807889752450442,
 'alpha': 0.05,
 'significant': True,
 'same_sign_sidos': 14,
 'same_sign_share': 0.8235294117647058}

In [4]:
stats.concentration

{'positive_sido_count': 7,
 'positive_youth_net_sum': 57245,
 'top3_sidos': ['경기', '서울', '인천'],
 'top3_values': [21053, 17207, 12472],
 'top3_share_of_positive_youth_net': 0.8862258712551314}

In [5]:
stats.sensitivity.loc[stats.sensitivity['sign_flips']]

,sido,net_20_39,net_20_34,sign_flips,typology_20_39
6,충남,292,-280,True,Family Settle
